[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/sphere_vortices/sphere_vortices.ipynb)

# Vortices on the Sphere

This example moves point vortices over a sphere by the flow they make together. A point vortex is modelled as a point with a circulation around it, the fluid swirling about it and irrotational everywhere else, and each vortex drifts with the flow of all the others. Equal vortices in a ring turn rigidly about its centre until, past a critical number, the ring breaks up; a pair of opposite vortices travels around the whole sphere, and two such pairs leapfrog as they go.

A vortex is modelled as the bivector of the turn about its point, the meet of two great circles, scaled by its circulation, so that negating both its orientation and its circulation leaves the same element. The flow at a place is one bivector, the sum of the vortices each weighted by its rate there, and that rate is the circulation over two pi times the squared norm of the join `A & B` of vortex and place, the great circle through both: the expression of the plane, where that norm is a distance, read here as a sine. Two time steps are built from rotors: one turns pairs of vortices rigidly by their exact rotors, keeping the impulse `vortices.sum()` to round-off; the other turns every vortex by the rotor of the flow at the middle of its step, `(flow * (-dt / 2)).exp() >> vortices`.

In Gibbs–Heaviside vector notation, each vortex is a unit vector $\mathbf x_k$ accompanied by its antipodal image $-\mathbf x_k$ of circulation $-\Gamma_k$, the velocity sums the two, $\dot{\mathbf x}_k = \frac{1}{4\pi} \sum_{j \ne k} \Gamma_j \, (\mathbf x_j \times \mathbf x_k) \left( \frac{1}{1 - \mathbf x_j \cdot \mathbf x_k} + \frac{1}{1 + \mathbf x_j \cdot \mathbf x_k} \right)$, and an explicit step renormalizes $\mathbf x_k$ back onto the sphere, while turning a pair rigidly about its axis $\mathbf n \propto \Gamma_j \mathbf x_j + \Gamma_k \mathbf x_k$ takes Rodrigues' formula $\mathbf x \cos\phi + (\mathbf n \times \mathbf x) \sin\phi + \mathbf n \, (\mathbf n \cdot \mathbf x)(1 - \cos\phi)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import jax
import numpy as np

from numga import NumpyContext, concatenate
from numga.algebras import Spherical3D as ga
from numga.backend.jax import JaxContext
from examples.util.animation import show_animation
from examples.fluids.sphere_vortices import render
from examples.fluids.sphere_vortices.scenarios import schedule

mv = NumpyContext(ga).multivector
# A point of the sphere as the meet of two great circles, oriented by the turn about it: scaled by its
# circulation, a vortex; summed over the vortices by their rates, the flow at a place.
Pivot = ga.gatype.bivector()                                                         # Bivector+

## Vortices as bivectors

A vortex is modelled as a bivector `A`, the meet of two great circles oriented by the turn about its point and scaled by its circulation, standing for the two antipodal points its rotor leaves fixed. A place on the sphere is a unit bivector `B` in the same way. The vortex turns every place about itself at a rate of its circulation over two pi times the squared norm of the join `A & B`, the great circle through the vortex and the place, whose norm is the sine of the angle between them and is the same from either of the vortex's two points.

Along each circle about the vortex the speed is then the circulation over two pi over the circle's radius, so the circulation around every such circle is the vortex's own, and the flow is irrotational everywhere except at the two points, where its vorticity sits with opposite signs, as a flow on a closed surface requires. The rate has the form it has in the plane, where the norm of the join of two points is their distance; only the algebra's signature differs.

In [ ]:
# The angles from the pole along a meridian of the places the flow is measured at.
angles = np.array([0.2, 1.0, np.pi / 2, 2.6])
n_places = len(angles)

In [ ]:
def sphere_points(colatitudes: np.ndarray, longitudes: np.ndarray) -> Pivot:
    """Points at the given angles from the pole and around it: the pole, the meet of the great circles x = 0 and y = 0, turned about the point (0, 1, 0), then about itself."""
    tilts = (mv.zx * (-colatitudes / 2)).exp()                                      # [...] Rotor+
    spins = (mv.xy * (-longitudes / 2)).exp()                                       # [...] Rotor+
    return spins >> (tilts >> mv.xy)                                                # [...] Pivot+


# One vortex of unit circulation at the pole, and places at the angles from it along a meridian.
pole = mv.xy                                                                         # [] Pivot+
places = sphere_points(angles, np.zeros(n_places))                                   # [n_places] Pivot+
# The commutator of a place with the flow there is the place's velocity; times the circumference of the circle
# about the vortex through the place, its speed gives the circulation around that circle.
speeds = places.commutator(pole / (2 * np.pi * (places & pole).norm_squared())).norm()   # [n_places] Scalar+
circulations = speeds * (2 * np.pi * np.sin(angles))                                 # [n_places] Scalar+

In [ ]:
print("circulation around circles about the vortex:", circulations.to_array())

## Rings of equal vortices

The flow at a place is one bivector, the sum of the vortices each weighted by its rate there, softened near each vortex by a core radius, and a vortex moves with the flow of the others. The midpoint step turns every vortex by the rotor of that flow at the middle of its step, found by a few sweeps of fixed-point iteration, so that each vortex moves along an arc of a circle. A step of a handful of vortices is a few hundred small operations, the rotor exponentials above all, and `jax.lax.scan` compiles a whole run of steps into one loop.

A ring of equal vortices around the point facing the viewer turns rigidly about that point, and every vortex here starts a random ten-thousandth off its place. In the plane, a ring of up to six such vortices is stable and a ring of seven is on the edge; on the sphere, twenty-five degrees from the pole, the rings of five and six turn on undisturbed and the ring of seven breaks up. Each vortex is drawn at both its points, on whichever side faces the viewer.

In [ ]:
# The rings' sizes, their angle from the pole and the vortices' core; the disturbance and its seed; the time step
# and the midpoint step's fixed-point sweeps; the rings' steps and the steps between frames, the view as elevation
# and azimuth in degrees, and the duration of a frame.
n_ring_vortices, ring_angle, ring_core = (5, 6, 7), np.radians(25.0), 0.02
disturbance, seed = 1e-4, 0
dt, n_sweeps = 0.05, 3
n_ring_steps, ring_stride = 320, 1
ring_view, frame_duration_ms = (70.0, -60.0), 40

In [ ]:
def flow(vortices: Pivot, core_radius: float) -> Pivot:
    """The flow of the other vortices at each vortex: their bivectors, each weighted by one over two pi times the squared norm of its join with the place, softened by the core radius."""
    n_vortices = vortices.shape[-1]
    units = vortices.normalized()                                                    # [..., n_vortices] Pivot+
    # The join of two points is the great circle through them; its norm is the sine of the angle between them.
    squared_sines = (units[..., :, None] & units[..., None, :]).norm_squared()       # [..., n_vortices, n_vortices] Scalar+
    # No vortex is moved by its own flow.
    rates = (1 - np.eye(n_vortices)) / (2 * np.pi * (squared_sines + core_radius**2))   # [..., n_vortices, n_vortices] Scalar+
    return (vortices[..., None, :] * rates).sum(axis=-1)                             # [..., n_vortices] Pivot+


def midpoint_step(vortices: Pivot, core_radius: float, dt: float, n_sweeps: int) -> Pivot:
    """The vortices one step later, each turned by the rotor of the flow of the others at the middle of its step, the middles found by fixed-point sweeps."""
    successors = vortices                                                            # [..., n_vortices] Pivot+
    for _ in range(n_sweeps):
        rotors = (flow((vortices + successors) / 2, core_radius) * (-dt / 2)).exp()   # [...] Rotor+
        successors = rotors >> vortices                                              # [...] Pivot+
    return successors

In [ ]:
def history(vortices: Pivot, step, n_steps: int) -> Pivot:
    """The vortices at every step, the steps compiled into one loop."""
    def advance(current: Pivot, _) -> tuple[Pivot, Pivot]:
        successor = step(current)
        return successor, successor

    # The vortices carried into JAX for the loop, and the history back.
    start = vortices.materialize(JaxContext(ga))                                     # [...] Pivot+
    _, successors = jax.lax.scan(advance, start, length=n_steps)                     # [n_steps, ...] Pivot+
    return concatenate([start[None], successors]).materialize(mv.context)            # [n_steps + 1, ...] Pivot+


rng = np.random.default_rng(seed)
rings = []
for n_vortices in n_ring_vortices:
    ring_places = sphere_points(np.full(n_vortices, ring_angle), 2 * np.pi * np.arange(n_vortices) / n_vortices)   # [n_vortices] Pivot+
    nudges = (mv.bivector(rng.normal(size=(n_vortices, 3))).normalized() * disturbance).exp()   # [n_vortices] Rotor+
    rings.append(history(nudges >> ring_places, lambda current: midpoint_step(current, ring_core, dt, n_sweeps),
                         n_ring_steps))                                              # [n_ring_steps + 1, n_vortices] Pivot+
ring_chords = [(ring[:, :, None] - ring[:, None, :]).norm_squared() for ring in rings]   # [n_ring_steps + 1, ...] Scalar+
show_animation(render.animate(rings, ring_view, ring_stride), "sphere_vortices_rings", frame_duration_ms)

## Travelling around the sphere

A pair of opposite vortices, side by side across the equator, travels along it around the whole sphere; two such pairs, the inner narrower, leapfrog as they go, taking turns passing through each other as in the plane. The positive vortices stay mirror images of the negative ones across the equator. This cell takes about ten seconds, most of it drawing the frames.

In [ ]:
# A dipole and two leapfrogging pairs on the first meridian, each vortex an angle from the equator, positive ones to
# the north, and their core; the steps and the steps between frames, and the view.
dipole_angles, dipole_circulations = np.array([0.3, -0.3]), np.array([1, -1])
frog_angles, frog_circulations = np.array([0.5, -0.5, 0.275, -0.275]), np.array([1, -1, 1, -1])
pair_core = 0.05
n_travel_steps, travel_stride = 640, 2
travel_view = (20.0, 0.0)

In [ ]:
dipole = sphere_points(np.pi / 2 - dipole_angles, np.zeros(len(dipole_angles))) * dipole_circulations   # [2] Pivot+
frogs = sphere_points(np.pi / 2 - frog_angles, np.zeros(len(frog_angles))) * frog_circulations         # [4] Pivot+
pair_histories = [history(vortices, lambda current: midpoint_step(current, pair_core, dt, n_sweeps), n_travel_steps)
                  for vortices in (dipole, frogs)]
# The mirror in the equator takes each positive vortex onto the negative vortex beside it.
mirror_images = mv.z >> pair_histories[1][:, ::2]                                    # [n_travel_steps + 1, 2] Pivot+
show_animation(render.animate(pair_histories, travel_view, travel_stride), "sphere_vortices_travelling", frame_duration_ms)

## Exact rotors of pairs

Two vortices alone keep the angle between them, so their sum stays the same while they move, and they turn rigidly about it. The pair step turns the vortices pair by pair, each pair by its exact rotor, through the rounds of a round-robin schedule for half the step and back for the other half.

The flow keeps the vortices' impulse, their sum `vortices.sum()`, for its symmetry under the turns of the sphere. The pairs' rotors move each pair rigidly and so keep the impulse exactly; the midpoint turns, each vortex by its own rotor, keep it only closely. Both steps are of second order, and on the leapfrogging pairs they part by a small fraction of how far the vortices travel.

In [ ]:
# The steps compared, from the start of the leapfrogging run above.
n_compared_steps = 100

In [ ]:
def pair_step(vortices: Pivot, partners: np.ndarray, core_radius: float, dt: float) -> Pivot:
    """The vortices one step later by the exact rotors of pairs, through the rounds of partners for half the step and back for the other half."""
    for partner in np.concatenate([partners, partners[::-1]]):
        others = vortices[..., partner]                                              # [...] Pivot+
        rates = 1 / (2 * np.pi * ((vortices.normalized() & others.normalized()).norm_squared() + core_radius**2))   # [...] Scalar+
        # Two vortices alone keep the angle between them and turn rigidly about their sum, weighted by the rate.
        rotors = ((vortices + others) * rates * (-dt / 4)).exp()                     # [...] Rotor+
        vortices = rotors >> vortices                                                # [...] Pivot+
    return vortices


partners = schedule(len(frog_angles))                                                # [3, 4]
pair_frogs = history(frogs, lambda current: pair_step(current, partners, pair_core, dt), n_compared_steps)   # [n_compared_steps + 1, 4] Pivot+
midpoint_frogs = pair_histories[1][:n_compared_steps + 1]                            # [n_compared_steps + 1, 4] Pivot+
pair_impulse_changes = (pair_frogs.sum(axis=-1) - frogs.sum(axis=-1)).norm()         # [n_compared_steps + 1] Scalar+
midpoint_impulse_changes = (midpoint_frogs.sum(axis=-1) - frogs.sum(axis=-1)).norm()   # [n_compared_steps + 1] Scalar+
parting = (pair_frogs[-1] - midpoint_frogs[-1]).norm()                               # [4] Scalar+
travel = (midpoint_frogs[-1] - midpoint_frogs[0]).norm()                             # [4] Scalar+
show_animation(render.animate([pair_frogs, midpoint_frogs], travel_view, 1), "sphere_vortices_compared", frame_duration_ms)

The left sphere shows the leapfrogging pairs stepped by the pairs' rotors and the right sphere the same pairs by the midpoint turns; over these steps the two stay close together.

In [ ]:
print("largest change of the impulse by the pairs' rotors:", pair_impulse_changes.max().to_array())
print("largest change of the impulse by the midpoint turns:", midpoint_impulse_changes.max().to_array())
print("largest parting of the two steps:", parting.max().to_array(), "largest travel:", travel.max().to_array())

In [ ]:
# checks
# The flow is irrotational away from the vortex's points.
assert ((circulations - 1).abs() < 1e-12).all()
# The rings of five and six keep their shape; the ring of seven breaks up.
ring_changes = [(chords - chords[0]).abs().max() for chords in ring_chords]
assert ring_changes[0] < 0.01 and ring_changes[1] < 0.05 and ring_changes[2] > 0.2
# The leapfrogging pairs stay mirror images across the equator.
assert ((mirror_images - pair_histories[1][:, 1::2]).norm() < 1e-8).all()
# The pairs' rotors keep the impulse to round-off, the midpoint turns closely; the steps part by a small fraction of
# the travel.
assert (pair_impulse_changes < 1e-10).all() and (midpoint_impulse_changes < 1e-2).all()
assert parting.max() < 0.05 * travel.max()